# Conversational Mode

Conversational mode is a behavioral pattern where the agent's primary activity is dialogue - asking questions, exploring ideas, clarifying ambiguity, and providing nuanced advice through back-and-forth exchange. The conversation itself is the value delivered; no specific task deliverable is required.

Unlike a basic chatbot (which is reactive and one-shot), conversational mode is active and directive: the agent steers the conversation toward clarity, insight, and useful outcomes. It manages dialogue as its core responsibility rather than treating conversation as a side-channel to task execution.

This mode is classified as dialogue-focused engagement. It can combine with any autonomy level:
- Conversational + chat: Pure dialogue, no tool use.
- Conversational + copilot: Agent gathers context via read-only tools to inform the conversation.
- Conversational + supervised: Agent can take actions mid-conversation but pauses for confirmation.

In [1]:
import os
import json
from dataclasses import dataclass, field
from typing import Optional
from langchain_openai import ChatOpenAI
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage

### Initialize the language model
Conversational mode benefits from a slightly higher temperature than task-execution modes. A temperature of `0.7` allows the agent to vary its phrasing naturally across turns - essential for dialogue that doesn't feel robotic - while still staying focused on the conversation goal.

In [2]:
llm = ChatOpenAI(model="gpt-4o-mini", api_key=os.getenv("OPENAI_API_KEY", "").strip(), temperature=0.7)

print("LLM initialized for conversational mode.")

LLM initialized for conversational mode.


## Tracking conversation state
What separates a conversational agent from a simple prompt-response loop is state. The agent needs to remember what has been established, what remains unclear, and what preferences the human has revealed. Without this, every response starts from scratch - the agent cannot build toward a conclusion.

We track four key elements:
- `established_facts`: What has been agreed or confirmed.
- `open_questions`: What remains unclear or unresolved.
- `human_preferences`: Constraints, values, or priorities the human has revealed.
- `conversation_phase`: Which stage the dialogue is in (exploring → clarifying → advising → concluding).

In [3]:
@dataclass
class ConversationState:
    """Tracks the evolving state of a conversational session.

    Attributes:
        topic: The subject or domain of the conversation.
        goal: What the conversation is trying to achieve.
        established_facts: Information confirmed or agreed by both parties.
        open_questions: Unresolved questions that need to be addressed.
        human_preferences: Constraints or priorities the human has revealed.
        conversation_phase: Current stage — exploring, clarifying, advising, or concluding.
        turn_count: Number of complete exchanges so far.
    """
    topic: str
    goal: str
    established_facts: list = field(default_factory=list)
    open_questions: list = field(default_factory=list)
    human_preferences: dict = field(default_factory=dict)
    conversation_phase: str = "exploring"
    turn_count: int = 0

The `conversation_phase` field is especially important: it tells the agent what kind of move to make next. In the early *exploring* phase, almost anything could be relevant. As the conversation narrows, the agent transitions to *clarifying* specific ambiguities, then *advising* once enough context has been gathered, and finally *concluding* with a summary and next steps.

## Building the conversational agent
The agent's core loop is: **receive message → select strategy → generate response**.

Strategy selection is the key differentiator from plain chat. The agent uses the current state to decide *what kind of move* to make:
- clarify: Too many unknowns — ask one focused question.
- explore: A few unknowns — probe the most important one.
- summarize: Periodic progress check (every 4–5 turns) to prevent drift.
- advise: Enough context — provide substantive, concrete advice.

The critical rule for clarification: ask only one question per turn. Listing multiple questions is one of the most common conversational failures - it overwhelms the human and signals the agent hasn't prioritized.

In [4]:
class ConversationalModeAgent:
    """An agent that manages a productive dialogue toward a defined goal.

    The agent tracks conversation state across turns, selects an appropriate response strategy based on how much clarity has been achieved, and generates responses calibrated to the current phase of the dialogue.
    """

    STRATEGY_INSTRUCTIONS = {
        "clarify": (
            "Ask exactly ONE focused clarifying question. "
            "Choose the question whose answer will most reduce uncertainty. "
            "Do not ask multiple questions — choose the single most important one."
        ),
        "explore": (
            "Share your current understanding of the key open question and "
            "invite the human to refine or correct it. Be specific about what "
            "you understand so far."
        ),
        "summarize": (
            "Briefly summarize what has been established in this conversation so far. "
            "State the key facts and the human's main preferences or constraints. "
            "Then confirm: 'Does this capture where we are?' before continuing."
        ),
        "advise": (
            "Provide substantive, specific advice based on what has been established. "
            "Be direct — give a clear perspective, not endless caveats. "
            "Acknowledge trade-offs explicitly. Where you are uncertain, say so briefly "
            "and then still provide your best recommendation."
        ),
    }

    def __init__(self, topic: str, goal: str):
        """Initialize the agent with a topic and conversation goal.

        Args:
            topic: The domain or subject of the conversation.
            goal: What the conversation is meant to achieve for the human.
        """
        self.state = ConversationState(topic=topic, goal=goal)
        self.history = []

    def respond(self, human_message: str) -> str:
        """Process a human message and return the agent's response.

        Args:
            human_message: The latest message from the human.

        Returns:
            The agent's response as a string.
        """
        self.state.turn_count += 1
        self.history.append(HumanMessage(content=human_message))

        strategy = self._determine_strategy()
        response = self._generate_response(strategy)

        self.history.append(AIMessage(content=response))
        return response

    def _determine_strategy(self) -> str:
        """Select the appropriate conversational move based on current state.

        Returns:
            Strategy name: 'clarify', 'explore', 'summarize', or 'advise'.
        """
        open_q = len(self.state.open_questions)
        turn = self.state.turn_count

        if open_q > 4:
            return "clarify"       # Too many unknowns — narrow down first
        elif open_q > 2:
            return "explore"       # Probe the most important open question
        elif turn > 1 and turn % 4 == 0:
            return "summarize"     # Periodic alignment check
        else:
            return "advise"        # Enough context — be substantive

    def _generate_response(self, strategy: str) -> str:
        """Generate a response appropriate to the selected strategy.

        Args:
            strategy: The conversational strategy to apply.

        Returns:
            The agent's response text.
        """
        system_prompt = f"""You are a thoughtful conversation partner operating in Conversational Mode.

CONVERSATION GOAL: {self.state.goal}
TOPIC: {self.state.topic}
CURRENT PHASE: {self.state.conversation_phase}
ESTABLISHED FACTS: {self.state.established_facts if self.state.established_facts else 'None yet'}
OPEN QUESTIONS: {self.state.open_questions if self.state.open_questions else 'None identified'}
HUMAN PREFERENCES: {self.state.human_preferences if self.state.human_preferences else 'None revealed'}

RESPONSE STRATEGY: {strategy.upper()}
{self.STRATEGY_INSTRUCTIONS[strategy]}

Respond naturally and conversationally. Keep responses concise (3-5 sentences) unless the strategy calls for a summary."""

        messages = [SystemMessage(content=system_prompt)] + self.history
        response = llm.invoke(messages)
        return response.content

    def update_state(self, new_facts=None, answered_questions=None,
                     new_questions=None, preferences=None):
        """Manually update conversation state between turns.

        Args:
            new_facts: Facts to add to established_facts.
            answered_questions: Questions to remove from open_questions.
            new_questions: New questions to add to open_questions.
            preferences: Preferences dict to merge into human_preferences.
        """
        if new_facts:
            self.state.established_facts.extend(new_facts)
        if answered_questions:
            self.state.open_questions = [
                q for q in self.state.open_questions if q not in answered_questions
            ]
        if new_questions:
            self.state.open_questions.extend(new_questions)
        if preferences:
            self.state.human_preferences.update(preferences)

The `_determine_strategy` method is deliberately simple - it uses counts and turn numbers rather than LLM inference. This keeps the control flow predictable and auditable. The LLM's role is generating the *content* of the response, not deciding *what kind* of response to make. Separating these concerns makes the agent's behavior easier to debug and tune.

## Demo: Career decision coaching conversation
The scenario below simulates a professional seeking guidance on a career transition. The conversation is pre-scripted on the human side (as it would be in real usage), while the agent responds dynamically.

We manually update the conversation state after each turn to illustrate how the agent's behavior shifts as context accumulates. In a production system, this state update would be driven by the LLM extracting facts and questions from each message automatically.

In [5]:
# Initialize agent for a career coaching conversation
agent = ConversationalModeAgent(
    topic="Career transition from software engineering to management",
    goal="Help the professional make a well-informed decision about accepting a management role"
)

# Seed the state with some open questions to trigger different strategies
agent.update_state(
    new_questions=[
        "What does the role actually involve day-to-day?",
        "Does the person value technical work or leadership?",
        "What are the financial implications?",
        "What happens to technical skills over time in management?",
        "What is the company culture around management?",
    ]
)

# Pre-scripted conversation turns
conversation = [
    "I've been offered a promotion to engineering manager. I'm not sure whether to take it.",
    "I've been an engineer for 6 years. I enjoy the technical work but my manager says I'm ready to lead.",
    "The pay increase is about 20%. But I'm worried I'll lose my technical edge over time.",
    "I think I'd miss solving hard problems myself. Though I do enjoy mentoring junior engineers.",
    "What would you recommend?",
]

print("=" * 60)
print("CAREER DECISION COACHING CONVERSATION")
print("=" * 60)

for i, user_msg in enumerate(conversation, 1):
    strategy = agent._determine_strategy()
    print(f"\nTurn {i} | Strategy: {strategy.upper()}")
    print(f"Human : {user_msg}")

    response = agent.respond(user_msg)
    print(f"Agent : {response}")

    # Simulate state updates based on what was revealed
    if i == 2:
        agent.update_state(
            new_facts=["6 years of engineering experience", "Manager believes they are ready"],
            answered_questions=["What does the role actually involve day-to-day?"],
            preferences={"values_technical_opinion": True}
        )
    elif i == 3:
        agent.update_state(
            new_facts=["20% pay increase offered"],
            answered_questions=["What are the financial implications?"],
            preferences={"concerned_about_technical_skills": True}
        )
    elif i == 4:
        agent.update_state(
            new_facts=["Enjoys mentoring junior engineers", "Values solving hard problems directly"],
            answered_questions=[
                "Does the person value technical work or leadership?",
                "What are the financial implications?"
            ]
        )
    elif i == 5:
        agent.state.conversation_phase = "advising"

print("\n" + "=" * 60)
print("CONVERSATION STATE SUMMARY")
print("=" * 60)
print(f"Established facts : {agent.state.established_facts}")
print(f"Open questions    : {agent.state.open_questions}")
print(f"Human preferences : {agent.state.human_preferences}")
print(f"Total turns       : {agent.state.turn_count}")

CAREER DECISION COACHING CONVERSATION

Turn 1 | Strategy: CLARIFY
Human : I've been offered a promotion to engineering manager. I'm not sure whether to take it.
Agent : That’s a significant opportunity! To help you weigh your decision, could you share what you value most in your work—do you find fulfillment in technical tasks, or are you more drawn to leadership and management responsibilities?

Turn 2 | Strategy: CLARIFY
Human : I've been an engineer for 6 years. I enjoy the technical work but my manager says I'm ready to lead.
Agent : It sounds like you have a solid foundation in engineering, and your manager sees potential in you for leadership. To further explore this, what do you envision your day-to-day responsibilities would be as an engineering manager? Understanding the specifics of the role might help clarify if it aligns with what you enjoy.

Turn 3 | Strategy: EXPLORE
Human : The pay increase is about 20%. But I'm worried I'll lose my technical edge over time.
Agent : A 20%

Notice how the agent's behavior shifts across the conversation. In the early turns, when many questions are open, it asks focused clarifying questions one at a time. As facts accumulate and open questions reduce, it transitions to exploring trade-offs and ultimately advising. The final turn explicitly requests a recommendation - the agent provides one directly, without deflecting into more questions.

## The funnel pattern
One of the most reliable structures for conversational mode is the **funnel pattern**: start broad to understand the full situation, then progressively narrow toward the specific question that most needs answering.

```
Turn 1:  "Tell me about your current situation"        → Broad context
Turn 2:  "Which aspect is most pressing right now?"    → Identify core challenge
Turn 3:  "What has prevented you from solving X?"      → Explore specific constraint
Turn 4:  Advise on the specific constraint               → Targeted advice
Turn 5:  "Does this address what you needed?"          → Confirm and close
```

The funnel works because it prevents premature advice. Jumping straight to "here's what you should do" before understanding the situation usually produces generic answers that miss the real constraint. The funnel earns the right to advise by first genuinely understanding.

In [6]:
def run_funnel_conversation(topic: str, goal: str, user_turns: list[str]) -> None:
    """Demonstrate the funnel pattern with a pre-scripted conversation.

    Args:
        topic: The subject of the conversation.
        goal: What the conversation aims to achieve.
        user_turns: Pre-scripted user messages.
    """
    system_prompt = f"""You are a strategic advisor using the Funnel Pattern.

GOAL: {goal}
TOPIC: {topic}

Funnel structure:
- Turn 1: Ask one broad question to understand the full situation
- Turn 2: Identify the single most pressing challenge
- Turn 3: Explore the specific constraint blocking progress
- Turn 4+: Provide targeted advice on the specific constraint

Keep each response to 2-3 sentences. Ask only ONE question per turn."""

    history = []
    print(f"\nFUNNEL CONVERSATION: {topic}")
    print("-" * 50)

    for i, user_msg in enumerate(user_turns, 1):
        history.append(HumanMessage(content=user_msg))
        messages = [SystemMessage(content=system_prompt)] + history
        response = llm.invoke(messages)
        history.append(AIMessage(content=response.content))

        phase = ["Broad", "Narrow", "Specific", "Advise"].pop(0) if i <= 4 else "Advise"
        print(f"\nTurn {i}")
        print(f"Human: {user_msg}")
        print(f"Agent: {response.content}")


# Example: API performance improvement conversation
run_funnel_conversation(
    topic="API performance improvement",
    goal="Help the engineer identify and address the root cause of API slowdowns",
    user_turns=[
        "Our API has been slow lately and users are complaining.",
        "The slowness is worst on the search endpoint, especially for large result sets.",
        "We haven't added any indexes since we launched, and the table has grown to 5 million rows.",
    ]
)


FUNNEL CONVERSATION: API performance improvement
--------------------------------------------------

Turn 1
Human: Our API has been slow lately and users are complaining.
Agent: What specific performance metrics or user feedback have you gathered to understand the nature of the slowdowns in your API?

Turn 2
Human: The slowness is worst on the search endpoint, especially for large result sets.
Agent: What do you think is the primary reason behind the slow performance of the search endpoint, such as data retrieval, processing time, or network latency?

Turn 3
Human: We haven't added any indexes since we launched, and the table has grown to 5 million rows.
Agent: What challenges or constraints have you encountered in implementing indexing or optimizing your database queries for the search functionality?


The funnel pattern prevents the agent from jumping to "add an index" in Turn 1 when the real situation might be much more complex (network latency, query design, caching, etc.). By Turn 3, the specific constraint (no indexes, large table) has been surfaced through dialogue - and the advice is now precise rather than generic.

## When to Use Conversational Mode
Conversational mode is the right choice when the *answer* cannot be determined without first understanding the *situation* more deeply. Use it when:

| Situation | Why Conversational Mode Fits |
|-----------|-----------------------------|
| Problem not yet fully defined | Human needs help clarifying what they actually want |
| Decision involves trade-offs | No single right answer; human must weigh competing values |
| Domain expertise needed | Human needs to understand a complex space before deciding |
| Creative exploration | Brainstorming or design thinking where the space is open |
| Emotional context matters | Human needs to feel heard before they can act on advice |
| High stakes / irreversible | Thorough dialogue before any action is taken |

**When NOT to use conversational mode**: If the task has a clear, bounded deliverable, use task execution Mode instead. Conversational mode applied to a clear task just adds unnecessary friction.

- **Conversational mode is active, not reactive**: The agent steers the dialogue toward a goal rather than simply responding to whatever the human says.
- **Strategy selection separates good conversational agents from chatbots**: Deciding *whether* to clarify, explore, summarize, or advise — based on state, not just the latest message — is what makes the agent purposeful.
- **One question per turn is a hard rule**: Asking multiple questions at once overwhelms the human and signals poor prioritization. Choose the single question whose answer reduces uncertainty the most.
- **The funnel pattern earns the right to advise**: By progressively narrowing from broad context to specific constraint, the agent ensures advice is targeted rather than generic.
- **Conversational mode pairs with any autonomy level**: The behavioral mode (dialogue management) is orthogonal to the autonomy mode (how much the agent can act independently).